# Demo 2 --- Governance at the decision point

A governed agent is bounded, not just capable. Every tool call passes a stack of gates before it runs; a call that fails a gate is refused, and a case that reaches the edge of the agent's authority escalates to a human. Each decision is written to a hash-chained audit log. This runs on the real harness.

In [ ]:
import json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
from forgeloop.agents.capstone import build_complaint_harness
from forgeloop.agents.core import Budget, BudgetTracker, TaskSpec
from forgeloop.agents.gms_backend import GMSPlausibilityGate

root = next((c for c in (Path('.'), Path('..'), Path('../..'), Path('../../code'), Path('../code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
cases = {c['id']: c for c in json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())}
harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')

def run(cid):
    task = TaskSpec(goal='handle complaint', inputs={'message': cases[cid]['message']})
    return harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))

## The gate stack runs on every call

Three gates guard each tool call: a syntax check that the call is well-formed, the banking policy engine (PII by regex, prompt-injection and prohibited-advice by a Qwen guard), and a trained GMS gate that scores the workflow transition against the policy graph. Only the last is a trained model; the rest is assembled scaffolding.

In [ ]:
gates = harness._executor._gates
for g in gates:
    print(f'{g.__class__.__name__:22s} trained={isinstance(g, GMSPlausibilityGate)}')

## A call refused at the gate

The message in `case-011` carries a Social Security number. The PII policy denies the very first tool call: the tool body never runs, the failed result names the gate that refused it, and the agent escalates rather than proceeding. Nothing unsafe reached the model or the log.

In [ ]:
traj = run('case-011')
print('message:', cases['case-011']['message'])
denied = next((r for r in traj.records if r.action.kind == 'tool_call'
               and r.observation and not r.observation.get('success')), None)
if denied:
    print('refused by gate:', denied.observation['error'])
esc = next((r for r in traj.records if r.action.kind == 'escalate'), None)
print('outcome        :', traj.final_state.status, '->', esc.action.reason if esc else '')

## An escalation on regulatory risk

The "unfair fee" message in `case-016` is well-formed and clears the gates, but `flag_regulatory` marks a UDAAP risk. The agent does not draft a reply that grants a remedy it cannot authorize; it escalates, naming the flag.

In [ ]:
traj = run('case-016')
print('message:', cases['case-016']['message'])
esc = next((r for r in traj.records if r.action.kind == 'escalate'), None)
print('outcome:', traj.final_state.status, '->', esc.action.reason if esc else '(drafted)')

## Every decision is in the audit chain

The harness logs one event per step and links them by hash, so the sequence of decisions can be replayed and verified. A refusal or an escalation is an auditable event, not a silent drop.

In [ ]:
print('audit events     :', len(harness.audit.events))
print('audit chain valid:', harness.audit.verify())

The audience sees the agent stopped twice --- once at a gate, once at a flag --- and both stops are recorded. That is the difference between an agent that is smart and one that is bounded.